In [ ]:

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim

from IPython.display import display, clear_output
from matplotlib import animation, rc
from IPython.display import HTML
import requests
import pickle, gzip, numpy

from matplotlib import colormaps

%matplotlib inline
#%matplotlib notebook
    

## Obtain Data

In [ ]:
# Get Data
if(not os.path.exists('mnist.pkl.gz')):
    r = requests.get('https://s3.amazonaws.com/img-datasets/mnist.pkl.gz')
    with open('mnist.pkl.gz', 'wb') as f:
        f.write(r.content)

#Load Data
f = gzip.open('mnist.pkl.gz', 'rb')
train_set, test_set = pickle.load(f,encoding='latin1')

#Show data
Xtr = torch.FloatTensor(train_set[0]/256.0)
Xtr = Xtr.reshape(Xtr.shape[0],Xtr.shape[1]*Xtr.shape[2])

Xte = torch.FloatTensor(test_set[0]/256.0)
Xte = Xte.reshape(Xte.shape[0],Xte.shape[1]*Xte.shape[2])



## Define Denoising Autoencoder Visualizer 

In [ ]:
def plot_im_array2(X,S,N,A,title):    
    I = np.ones(((A*(1+S),A*(1+S))))*max(X.flatten())
    k=0
    for i in range(A):
        for j in range(A):
            I[i*(S)+i:(i+1)*S+i,j*S+j:(j+1)*S+j] = X[k,:].reshape((S, S))
            k=k+1
            if(k==N): break
        if(k==N): break
            
    plt.imshow(I, cmap=plt.cm.gray,interpolation=None)
    plt.colorbar()
    plt.xticks(())
    plt.yticks(())
    plt.title(title)

def dae_evaluator(X,XN,R,H,Y,err,ds,epoch=None):

    if(Y is not None):
        s1,s2=2,2
        plt.figure(figsize=[10,8])
    else:
        s1,s2=1,3
        plt.figure(figsize=[16,4])

    plt.subplot(s1,s2,1)
    plot_im_array2(XN[:100,:].numpy(),28,100,10,f"Noisy {ds} Samples")
    plt.clim(0,1)

    plt.subplot(s1,s2,2)
    if(epoch is None):
        plot_im_array2(R[:100,:].numpy(),28,100,10,f"{ds} Reconstructions Err: {err:0.4f}")
    else:
        plot_im_array2(R[:100,:].numpy(),28,100,10,f"{ds} Reconstructions Epoch: {epoch} Err: { err:0.4f}")
    plt.clim(0,1)

    plt.subplot(s1,s2,3)
    plot_im_array2(X[:100,:].numpy(),28,100,10,f"True {ds} Samples")
    plt.clim(0,1)

    if(Y is not None and H.shape[1]==2):
        plt.subplot(s1,s2,4)
        for c in range(10):
            plt.plot(H[Y==c,0],H[Y==c,1],'o', label="%d"%c,alpha=1)
        plt.legend(location="upperright")
        plt.grid(True)

    if(Y is not None and H.shape[1]>2):
        Ysub = Y[:100]
        Hsub = H[:100,:]
        plt.subplot(s1,s2,4)
        ind = np.argsort(Ysub)
        counts = np.bincount(Ysub)
        boundaries = np.cumsum(counts)
        Hnorm = (Hsub[ind,:]-np.mean(H,axis=0,keepdims=True))/np.std(H,axis=0,keepdims=True)
        plt.imshow(np.flipud(Hnorm),cmap=colormaps["bwr"],interpolation="nearest",aspect="auto")
        plt.gca().set_yticks(boundaries - counts[0]-0.5)
        plt.gca().set_yticklabels(np.flip(np.arange(10)) ) 
        plt.colorbar()
        plt.grid(True,axis='y', color='k', linestyle='-', linewidth=1)
        cmax = np.percentile(np.abs(Hnorm),99)
        plt.clim(-cmax,cmax)


## Define Denosing Autoencoder Training

In [ ]:
class noise_model():
    def __init__(self, noise_level, noise_type):
        self.noise_level = noise_level
        self.noise_type=noise_type

    def add_noise(self,X):

        if(self.noise_type=="normal"):
           return torch.clip(X + self.noise_level*torch.randn(X.size()),0.01,0.99)
        
        if(self.noise_type=="sp"):
            mask = (torch.rand(X.size()) < self.noise_level).type(torch.float)
            return X*(1.0-mask) + torch.rand(X.size()) * mask

def dae_trainer(X, model, noise_model, batch_size=100, num_epochs = 100, base_lr = 0.001, max_lr=0.01, weight_decay=0, Y=None):

    N = X.size()[0]
    B = int(np.ceil(N/batch_size))

    device     = "cuda" if torch.cuda.is_available() else "cpu" #Set device to gpu or cpu
    optimizer  = optim.Adam(model.parameters(), lr=1,weight_decay=weight_decay)

    scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor=max_lr, end_factor=base_lr, total_iters=B*100)
    train_loss = []
    X          = X.to(device)

    model.train() # training mode
 
    old_total_loss=np.inf
    for epoch in range(num_epochs): #(loop for every epoch)

        total_loss=0
        for iter in range(B):
            start = iter*batch_size
            end   = min(N,(iter+1)*batch_size)
            thisX = X[start:end,...]

            optimizer.zero_grad()                    #Zero the gradient
            XN =  noise_model.add_noise(thisX)       #Add noise
            R = model.forward(XN)                    #Compute reconstructions
            loss = torch.mean((thisX-R)**2)         #Compute loss
            #loss = -1*torch.mean(thisX * torch.log(R) + (1-thisX) * torch.log(1-R) )                      #Compute loss
            loss.backward()                          #Compute the gradient of the loss
            optimizer.step()                         #Take a step
            total_loss+=loss.item()/B
            scheduler.step()

        if(epoch%1==0):
            clear_output(wait=True)
            with torch.no_grad():
                model.eval()
                Xsub = X[range(0,10000,10),...]
                XN =  noise_model.add_noise(Xsub)
                R = model.forward(XN)              
                H = model.encode(Xsub).detach().numpy()
                Ysub = Y[range(0,10000,10)]
                model.train()

            train_loss.append(loss.item())           # update running loss and error
            #print(f"[Train Epoch: {epoch}] Loss: { train_loss[-1]:.8f}")
            dae_evaluator(Xsub,XN,R,H,Ysub,total_loss,"Train",epoch=epoch)

            plt.show()

        if(epoch > 0):
            if(np.abs(total_loss - old_total_loss )/old_total_loss < 1e-4):
                break
        
        old_total_loss = total_loss

            


## Visualize Clean and Noisy Training Data

In [ ]:
plt.figure(figsize=[15,4])
plt.subplot(1,3,1)
plot_im_array2(Xtr[range(0,50000,500),:].numpy(),28,100,10,"Train Data Samples")

plt.subplot(1,3,2)
nm = noise_model(0.1,"normal")
plot_im_array2(nm.add_noise(Xtr[range(0,50000,500),:]).numpy(),28,100,10,"Normal Noise Train Data Samples")

plt.subplot(1,3,3)
nm = noise_model(0.1,"sp")
plot_im_array2(nm.add_noise(Xtr[range(0,50000,500),:]).numpy(),28,100,10,"SP Noise Train Data Samples")
plt.show()

Ytr = train_set[1]
Yte = test_set[1]

## Define, Learn and Visualize Model 

In [ ]:
class linear_dae(nn.Module):
    def __init__(self,D,K):
        super(linear_dae, self).__init__()
        self.fc1 = nn.Linear(D, K)
        self.fc2 = nn.Linear(K, D)
    
    def encode(self,X):
        return(self.fc1(X))
    
    def decode(self,H):
        return self.fc2(H)

    def forward(self, X):
        H = self.encode(X)
        R = self.decode(H)
        return(R)

nm   = noise_model(0.1,"sp")
ldae = linear_dae(Xtr.shape[1],50)
dae_trainer(Xtr, ldae, nm, num_epochs = 100, batch_size=256,  Y=Ytr, base_lr = 0.0001, max_lr = 0.001, weight_decay=1e-6)

In [ ]:
class mlp_dae(nn.Module):
    def __init__(self,D,K1,K2):
        super(mlp_dae, self).__init__()

        f = nn.Tanh()

        self.encode = nn.Sequential(
            nn.Linear(D, K1), f,
            nn.Linear(K1, K2), f
        )

        self.decode = nn.Sequential(
            nn.Linear(K2, K1), f,
            nn.Linear(K1, D), nn.Sigmoid()
        )        

    def forward(self, X):
        H = self.encode(X)
        R = self.decode(H)
        return(R)
        
nm   = noise_model(0.1,"sp")
nldae = mlp_dae(Xtr.shape[1],100,50)
dae_trainer(Xtr, nldae, nm, batch_size=256, num_epochs = 100, base_lr = 0.0001, max_lr = 0.001, Y=Ytr, weight_decay=1e-6)